# Qwen3.5-4B RotQuant LoRA-QAT on CUDA

This notebook is a reproducible **text-backbone quality experiment** for `unsloth/Qwen3.5-4B` on a high-memory NVIDIA GPU. It starts from the current 4-bit RotQuant candidate, trains propagation-aware butterfly rotations and scales block-by-block, then performs rank-4 LoRA distillation against source-model logits.

The local Apple Silicon result was **17.8463 source PPL vs 18.9015 4-bit FWHT PPL** on 32 WikiText-2 samples. The CUDA source baseline produced below is authoritative for comparison with the CUDA-trained candidate.

## Goal and key assumptions

- Primary gate: no more than **10% perplexity degradation** from the matched CUDA source model; the stretch gate is 5%.
- Only decoder-block language linears are quantized. The vision tower, tied output head, and `linear_attn.in_proj_a/b` remain in source precision.
- `patch.fallback=true` caches dequantized FP16 weights during QAT for speed. It is suitable for this quality experiment on a high-memory GPU, but its peak VRAM is **not** the packed deployment footprint.
- The current repository records trained quality and exact byte accounting in JSON, but does not yet export a reloadable packed checkpoint. Download the results archive before the Colab runtime ends.
- `REPO_REF` must point to a pushed branch containing `rotquant/block_train.py` and `configs/qwen35_4b_lora_qat_cuda.yaml`.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"  # Change to the branch containing the Qwen LoRA-QAT files.
REPO_DIR = Path("/content/rotquant")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_lora_qat_cuda.yaml")
RESULT_ROOT = Path("/content/rotquant_colab_results")

EVAL_SEQ_LEN = 256
EVAL_MAX_SAMPLES = 32
TARGET_RELATIVE_PPL = 0.10
STRETCH_RELATIVE_PPL = 0.05

# Run All performs the matched baseline and bounded smoke trial. Flip this
# only after the smoke trial succeeds to permit the expensive full run.
RUN_FULL = False
DOWNLOAD_RESULTS = True
TRACKIO_SPACE_ID = None  # Optional, e.g. "your-hf-username/trackio".

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print({"repo_ref": REPO_REF, "model": MODEL_ID, "run_full": RUN_FULL})

### 1. Verify the CUDA runtime

Use a GPU runtime. The intended RTX PRO 6000 Server Edition has ample memory for cached-weight QAT; the notebook warns below 40 GiB.

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name}")
print(f"VRAM: {vram_gib:.1f} GiB | torch={torch.__version__} | CUDA={torch.version.cuda}")
if vram_gib < 40:
    print("WARNING: below 40 GiB; cached fallback may OOM. Use a larger GPU or set patch.fallback=false.")
subprocess.run(["nvidia-smi"], check=True)

### 2. Fetch RotQuant and install compatible libraries

The installation preserves Colab's CUDA-enabled PyTorch build and upgrades Transformers for Qwen3.5 multimodal loading.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    print(f"Reusing existing checkout at {REPO_DIR}; delete it to clone afresh.")

required_paths = [
    REPO_DIR / "rotquant/block_train.py",
    REPO_DIR / CONFIG_RELATIVE_PATH,
    REPO_DIR / "scripts/run_experiment.py",
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, (
    "REPO_REF does not contain the required Qwen LoRA-QAT changes: "
    + ", ".join(missing)
)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print(f"Using commit {commit}")

In [ ]:
runtime_packages = [
    "transformers>=5.9,<6",
    "datasets>=4.8",
    "accelerate",
    "safetensors",
    "sentencepiece",
    "scipy",
    "pyyaml",
    "pandas",
    "huggingface_hub",
    "trackio",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

# Speed up the trainer's float32 block-reconstruction matmuls on modern NVIDIA GPUs.
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

import transformers
from transformers import AutoModelForMultimodalLM
print(f"transformers={transformers.__version__}; multimodal loader available")

### 3. Validate the experiment configuration

In [ ]:
import yaml

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
with config_path.open() as handle:
    experiment_config = yaml.safe_load(handle)

assert experiment_config["model"] == MODEL_ID
assert experiment_config["device"] == "cuda"
assert experiment_config["quant"]["bits"] == 4
assert experiment_config["patch"]["fallback"] is True
assert experiment_config["patch"]["train_rotation"]["distill_lora_rank"] == 4
assert experiment_config["patch"]["include"] == ["model.language_model.layers."]
print(yaml.safe_dump(experiment_config, sort_keys=False))

## Steps

### 4. Define the bounded runner

Each phase writes to its own directory. A failed process raises immediately instead of silently reading a stale result.

In [ ]:
import json
import shlex
from typing import Iterable

def run_trial(trial_name: str, overrides: Iterable[str]):
    output_dir = RESULT_ROOT / trial_name
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        str(REPO_DIR / "scripts/run_experiment.py"),
        str(config_path),
        "--output-dir",
        str(output_dir),
    ]
    for override in overrides:
        command.extend(["--set", override])
    print("Running:", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)
    candidates = sorted(output_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
    assert candidates, f"No result JSON was written to {output_dir}"
    with candidates[-1].open() as handle:
        payload = json.load(handle)
    print(f"Result: {candidates[-1]}")
    return payload

matched_eval_overrides = [
    f"eval.ppl.seq_len={EVAL_SEQ_LEN}",
    f"eval.ppl.max_samples={EVAL_MAX_SAMPLES}",
]

### 5. Establish the matched CUDA source baseline

Do not compare the trained CUDA candidate only with the earlier MPS number; this run controls for backend and dtype.

In [ ]:
baseline_result = run_trial(
    "cuda_source_baseline",
    ["patch.enabled=false", *matched_eval_overrides],
)
baseline_ppl = baseline_result["metrics"]["ppl_wikitext2"]
print(f"Matched CUDA source PPL: {baseline_ppl:.4f}")

### 6. Run a smoke LoRA-QAT trial

This still touches all 32 language blocks, but uses one block-training step and two distillation steps. Its four-sample PPL is a health check only—not the final quality comparison.

In [ ]:
smoke_overrides = [
    "patch.train_rotation.steps=1",
    "patch.train_rotation.train_batches=1",
    "patch.train_rotation.validation_batches=1",
    "patch.train_rotation.selection_batches=1",
    "patch.train_rotation.early_stopping_patience=0",
    "patch.train_rotation.distill_steps=2",
    "patch.train_rotation.distill_train_batches=1",
    "patch.train_rotation.distill_validation_batches=1",
    "patch.train_rotation.distill_selection_batches=1",
    f"eval.ppl.seq_len={EVAL_SEQ_LEN}",
    "eval.ppl.max_samples=4",
]
smoke_result = run_trial("lora_qat_smoke", smoke_overrides)
smoke_metrics = smoke_result["metrics"]
print({
    "ppl_health_check": smoke_metrics.get("ppl_wikitext2"),
    "quantized_layers": smoke_metrics.get("n_quant_layers"),
    "distillation_accepted": smoke_metrics.get("distillation", {}).get("accepted"),
    "peak_training_vram_gib": smoke_metrics.get("peak_vram_bytes_patch", 0) / 2**30,
})

### 7. Run the full 4-bit LoRA-QAT experiment

After the smoke trial succeeds, set `RUN_FULL = True` in the parameter cell and run this cell. The config uses 12 block steps, disjoint 2/2/2 block calls, 12 distillation steps, and disjoint 2/1/1 teacher-logit calls.

In [ ]:
if RUN_FULL:
    full_result = run_trial("lora_qat_full", matched_eval_overrides)
else:
    full_result = None
    print("Full run is gated. Set RUN_FULL = True after reviewing the smoke output.")

## Checks

### 8. Evaluate the quality and complete-model size gates

In [ ]:
import pandas as pd

comparison = None
if full_result is None:
    print("Run the full experiment before evaluating the release gates.")
else:
    from huggingface_hub import hf_hub_download

    full_metrics = full_result["metrics"]
    trained_ppl = full_metrics["ppl_wikitext2"]
    relative_ppl = trained_ppl / baseline_ppl - 1.0

    index_path = hf_hub_download(MODEL_ID, "model.safetensors.index.json")
    with open(index_path) as handle:
        source_weight_bytes = json.load(handle)["metadata"]["total_size"]
    quantized_source_bytes = full_metrics["fp16_weight_bytes"]
    deployed_quantized_bytes = full_metrics.get(
        "packed_plus_auxiliary_bytes", full_metrics["packed_weight_bytes"]
    )
    estimated_model_bytes = (
        source_weight_bytes - quantized_source_bytes + deployed_quantized_bytes
    )
    size_reduction = 1.0 - estimated_model_bytes / source_weight_bytes

    comparison = pd.DataFrame([
        {
            "variant": "source CUDA",
            "ppl_wikitext2": baseline_ppl,
            "relative_ppl": 0.0,
            "estimated_weight_GB": source_weight_bytes / 1e9,
        },
        {
            "variant": "4-bit RotQuant + LoRA-QAT",
            "ppl_wikitext2": trained_ppl,
            "relative_ppl": relative_ppl,
            "estimated_weight_GB": estimated_model_bytes / 1e9,
        },
    ])
    display(comparison.style.format({
        "ppl_wikitext2": "{:.4f}",
        "relative_ppl": "{:+.2%}",
        "estimated_weight_GB": "{:.3f}",
    }))
    print(f"10% quality gate: {'PASS' if relative_ppl <= TARGET_RELATIVE_PPL else 'FAIL'}")
    print(f"5% stretch gate: {'PASS' if relative_ppl <= STRETCH_RELATIVE_PPL else 'FAIL'}")
    print(f"Estimated complete-model reduction: {size_reduction:.2%}")
    print(f"LoRA retained: {full_metrics.get('distillation', {}).get('lora_retained')}")
    print(f"Adapter bytes: {full_metrics.get('adapter_parameter_bytes', 0):,}")

### 9. Optionally publish the final summary to Trackio

Set `TRACKIO_SPACE_ID` to your Hugging Face Space, such as `username/trackio`, to retain the final comparison alongside future trials. RotQuant's detailed per-block trace remains in the cell output and result JSON.

In [ ]:
if TRACKIO_SPACE_ID and full_result is not None:
    import trackio

    trackio.init(
        project="rotquant-qwen35",
        name="qwen35-4b-4bit-lora-qat",
        space_id=TRACKIO_SPACE_ID,
        config={
            "model": MODEL_ID,
            "bits": 4,
            "lora_rank": 4,
            "eval_seq_len": EVAL_SEQ_LEN,
            "eval_max_samples": EVAL_MAX_SAMPLES,
        },
    )
    trackio.log({
        "source_ppl": baseline_ppl,
        "trained_ppl": trained_ppl,
        "relative_ppl": relative_ppl,
        "estimated_model_bytes": estimated_model_bytes,
        "size_reduction": size_reduction,
    })
    trackio.finish()
    print(f"Published summary to {TRACKIO_SPACE_ID}")
else:
    print("Trackio publishing skipped.")

### 10. Download the experiment record

This archive contains provenance, configuration, training-selection statistics, byte accounting, and perplexity. It does not contain a reloadable model checkpoint yet.

In [ ]:
import shutil

archive_base = Path("/content/qwen35_rotquant_lora_qat_results")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULT_ROOT))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1e6:.2f} MB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))

## Next steps

1. Accept this candidate only if the matched 32-sample CUDA run passes the selected perplexity gate and the held-out distillation gate retains LoRA.
2. Repeat seeds 1 and 2 before treating the result as stable.
3. If 4-bit passes comfortably, try the same pipeline at 3-bit; otherwise keep 4-bit.
4. Implement packed checkpoint serialization and a serving kernel before claiming actual deployment-memory or throughput savings.